# Ask Hestia · Assignment 8 RAG experiments
Five household questions, three chunking strategies and three embedding encoders. Each of the 45 requests uses the same Qwen2 generator. The tables below read the measured runs and the fact-by-fact answer review. Set the inference switch to repeat the experiment on your hardware.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
root = Path.cwd()
if (root / "corpus.py").is_file() and root.name == "a8":
    root = root.parent
sys.path.insert(0, str(root))
from a8.corpus import load_chunks, load_queries, STRATEGIES
from a8.person3_experiments import OUT, run
from a8.person3_analysis import evaluation_table, summary, SIZES
pd.DataFrame([{"strategy": s, "full chunks": len(load_chunks(s, "full")),
               "subset chunks": len(load_chunks(s, "small"))} for s in STRATEGIES])

,strategy,full chunks,subset chunks
0,fixed,30,15
1,overlap,20,10
2,hybrid,20,10


### Generation and retrieval
The query is embedded and matched to indexed handbook chunks by cosine similarity. The generator receives its system instruction, the unchanged top three passages in rank order, and the original question. The answer key is used only after decoding. Model loading, index construction, one warm-up per configuration and scoring are outside the measured latency.

In [2]:
from a8.generation import MODEL, REVISION, SYSTEM, MAX_NEW_TOKENS
print(MODEL, REVISION)
print(SYSTEM)
print(f"Greedy decoding, maximum {MAX_NEW_TOKENS} new tokens.")
display(pd.DataFrame(load_queries())[["id", "query"]])

Qwen/Qwen2-0.5B-Instruct c540970f9e29518b1d8f06ab8b24cba66ad77b6d
You answer questions about the Hestia household handbook. Use only the cited passages. Answer all parts of the question directly, preserving the names, quantities and conditions in the handbook. If a detail is missing, say that the handbook does not specify it.
Greedy decoding, maximum 128 new tokens.


,id,query
0,Q1,"I finished closing the kitchen, but the dishwa..."
1,Q2,The store ran out of an item on our shared gro...
2,Q3,I need the shared cordless drill this afternoo...
3,Q4,The bathroom tap has a slow drip even with the...
4,Q5,A friend is staying overnight. What needs to b...


In [3]:
RUN_INFERENCE = False
if RUN_INFERENCE:
    # Fresh raw results are kept separate from the reviewed reference experiment.
    for size in SIZES:
        run(size, OUT.parent / "repeat" / size)

### Evaluation table
Context quality is the percentage of required evidence quotes found in the retrieved texts. Answer quality is the percentage of fully conveyed required facts after deductions for unsupported instructions. See person3/RUBRIC.md and results/person3/reviews.csv for grading decisions. End-to-end timing measures an actual request.

In [4]:
rows = evaluation_table()
table = pd.DataFrame(rows)
display(table[["size", "strategy", "query_id", "retrieved_context_quality", "answer_quality",
               "end_to_end_ms", "notes"]].round(2).style.hide(axis="index"))

size,strategy,query_id,retrieved_context_quality,answer_quality,end_to_end_ms,notes
small,fixed,Q1,100.000000,66.670000,4729.640000,Completes the card and defers unloading to the morning resident. Assumes the tag is already attached.
small,fixed,Q2,100.000000,0.000000,1638.910000,Omits both required actions and suggests closing the grocery entry.
small,fixed,Q3,100.000000,0.000000,6005.480000,"Never identifies the blue bin or gives the complete checkout record. Directs the reader to look up the location on the card, log a check time, and log location/status fields."
small,fixed,Q4,100.000000,0.000000,7032.720000,Identifies the Plumbing card but never says to reuse it or records the three details. Makes the photo mandatory and conditions notification on the card being unavailable. Output ends mid-sentence.
small,fixed,Q5,100.000000,20.000000,3997.310000,"Gives a sheet set. Leaves out exact towel and carafe placement, the lamp check and the completion rule."
small,overlap,Q1,100.000000,100.000000,6361.770000,"All three actions appear, although the answer describes them in the first person."
small,overlap,Q2,100.000000,0.000000,1563.710000,Quantity alone does not say to leave the item unchecked or write Out of stock.
small,overlap,Q3,100.000000,0.000000,5017.360000,"Omits the borrower name, bin and closure. Sends the reader to the checkout card for the location and to appliance upkeep for the return."
small,overlap,Q4,100.000000,75.000000,6174.520000,"Reuses the existing card, records all three details and notifies through it. Does not name the Plumbing card on the Maintenance board."
small,overlap,Q5,100.000000,100.000000,7384.420000,"All required quantities, placements, lamp check and completion condition appear before the final sentence cuts off."


### Inspect an individual request
Change the row ID to inspect the exact prompt, full ranked chunks, generated answer and reference for any of the 45 requests.

In [5]:
row_id = "medium/full/hybrid/Q3"
size = row_id.split("/")[0]
records = [json.loads(line) for line in (OUT / size / "rows.jsonl").read_text().splitlines()]
example = next(r for r in records if r["row_id"] == row_id)
print(example["prompt"])
print("\nGenerated answer:", example["answer"])
print("\nReference:", next(q["expected_answer"] for q in load_queries() if q["id"] == example["query_id"]))

<|im_start|>system
You answer questions about the Hestia household handbook. Use only the cited passages. Answer all parts of the question directly, preserving the names, quantities and conditions in the handbook. If a detail is missing, say that the handbook does not specify it.<|im_end|>
<|im_start|>user
Handbook passages:
[hybrid-inventory-01]
The shared tool inventory identifies equipment that belongs to the household and its normal storage location. Small hand tools are arranged in labeled bins so a resident can tell whether a missing item is being used or was put away elsewhere. The inventory card includes the tool name, its home bin, and any accessories that travel with it. Before starting a project, check the card and confirm that the necessary pieces are present. Residents leave personal equipment in their own storage unless they have agreed to share it. This keeps household availability records meaningful for everyone.

Before borrowing the shared cordless drill, add your nam

### Embedding comparison

In [6]:
display(pd.DataFrame(summary(rows)).round(2))
display(Markdown((root / "a8" / "person3" / "COMPARISON.md").read_text()))

,size,dimension,runs,mean_context,mean_answer,rank_one,median_retrieval_ms,median_request_ms
0,small,384,15,100.0,37.44,15,9.10,4729.64
1,medium,768,15,100.0,33.33,15,27.82,4114.95
2,large,1024,15,100.0,41.89,15,84.48,3492.91


# Step 2.3 · Does a larger embedding improve Hestia's answers?

These are matched runs of five handbook questions under fixed, overlap and hybrid chunking. Only the embedding encoder and its specified query formatting vary within a matched query/strategy comparison. The full corpus, top three, generator, prompt format, greedy decoding, output limit and CPU thread count are fixed. Each mean gives each of the 15 rows equal weight. Scores are on a 0–100 scale under the shared rubric.

| Embedding model | Dimensions | Runs | Evidence coverage | First supporting hit at rank 1 | Mean answer quality | Median retrieval | Median full request |
|---|---:|---:|---:|---:|---:|---:|---:|
| all-MiniLM-L6-v2 | 384 | 15 | 100% | 15/15 | 37.44 | 9.1 ms | 4.73 s |
| all-mpnet-base-v2 | 768 | 15 | 100% | 15/15 | 33.33 | 27.8 ms | 4.11 s |
| bge-large-en-v1.5 | 1,024 | 15 | 100% | 15/15 | 41.89 | 84.5 ms | 3.49 s |

**Retrieval quality.** For every encoder, the first hit referenced a supporting paragraph ID, and the three passages together contained every required quote. A fixed-length chunk can contain only part of a paragraph. Inspecting the first hit alone, it contained all required quotes on 11/15 small, 10/15 medium and 10/15 large requests. On the fixed grocery question, the first hit contained none of the required quote for all three encoders. With these five questions there was no measurable gain in those two retrieval metrics from increasing dimension. The identities of later passages could still change. Those passages are part of the generator's input.

**Answer quality.** BGE had the highest mean, while MPNet scored below MiniLM. On the out-of-stock question with overlapping chunks, the small encoder's answer just said to record the quantity. The medium and large runs both said to leave the original item unchecked and write “Out of stock,” earning 100. With fixed chunks on the same question, all three scored zero despite retrieving the rule. The prompt and raw passages for each case are in the JSONL files.

**Did larger always win?** No. On the kitchen handoff with fixed chunks, the small run scored 66.67, the medium 0 and the large 33.33. For preparing a guest room with hybrid chunks, all three covered every required fact. Embedding dimension alone cannot explain these observations: the encoders differ in training, and the generator can overlook or misuse evidence. The 128-token cap was reached on 11 of 45 requests; the reviews score only what the model actually returned.

**Latency.** Median retrieval latency grew with encoder size on this CPU. End-to-end medians did not follow the same order. The full request includes variable-length generation. There is one timing per configuration/query after warm-up, models were run in sequence, and five questions from one handbook are insufficient to establish a general speed or quality ranking.

The per-answer scores are an AI-assisted review of actual generated text. The matched facts, deductions, review notes and evidence fingerprints are in results/person3/reviews.csv. The full evaluation table includes the retrieved texts and actual end-to-end latency for each request.


## Person 2 completed Colab matrix and scaling

The following is a **separate hosted CPU run**, not additional rows to average into the matched 45-row cross-encoder comparison above. It completes the 15 Person 2 small-model RAG rows and the 10 small/full hybrid scaling requests. Generator revision, prompt, greedy settings and k=3 are fixed. Reviews are AI-assisted; student signoff is not recorded. All raw prompts, contexts and answers are preserved.

In [7]:
from a8.person2_analysis import evaluation_table as person2_evaluation
person2_run = root / 'a8/results/person2/completed-2026-11-09'
person2_table = pd.DataFrame(person2_evaluation(person2_run))
assert len(person2_table)==25
person2_matrix = person2_table[person2_table.experiment=='matrix']
display(person2_matrix[['embedding_model','strategy','query_id','retrieved_context_quality',
                       'answer_quality','retrieval_ms','generation_ms','end_to_end_ms','notes']])

,embedding_model,strategy,query_id,retrieved_context_quality,answer_quality,retrieval_ms,generation_ms,end_to_end_ms,notes
0,sentence-transformers/all-MiniLM-L6-v2,fixed,Q1,100.0,66.67,86.089021,29862.879253,29949.150205,Completes the existing closing card and assign...
1,sentence-transformers/all-MiniLM-L6-v2,fixed,Q2,100.0,0.00,54.226600,11467.331860,11521.662689,Neither leaves the original unchecked nor writ...
2,sentence-transformers/all-MiniLM-L6-v2,fixed,Q3,100.0,0.00,54.017444,36288.608385,36342.723434,"Omits the blue bin, complete borrower/tool/exp..."
3,sentence-transformers/all-MiniLM-L6-v2,fixed,Q4,100.0,0.00,41.424276,39983.064480,40024.632925,"Names Plumbing on the Maintenance board, but d..."
4,sentence-transformers/all-MiniLM-L6-v2,fixed,Q5,100.0,20.00,37.882141,25888.567141,25926.573542,A clean sheet set conveys one set. Towels/cara...
5,sentence-transformers/all-MiniLM-L6-v2,overlap,Q1,100.0,100.00,35.230797,41076.742072,41112.127634,All three required handoff actions appear desp...
6,sentence-transformers/all-MiniLM-L6-v2,overlap,Q2,100.0,0.00,35.707337,14612.830686,14648.634584,Quantity still needed is supported background ...
7,sentence-transformers/all-MiniLM-L6-v2,overlap,Q3,100.0,0.00,34.542153,33512.464598,33547.190222,"No complete required fact: borrower name, exac..."
8,sentence-transformers/all-MiniLM-L6-v2,overlap,Q4,100.0,75.00,49.868729,36946.307874,36996.293516,"Correctly reuses the card, records room/time/c..."
9,sentence-transformers/all-MiniLM-L6-v2,overlap,Q5,100.0,100.00,47.770024,45501.037847,45548.975651,"All quantities, placements, lamp check and com..."


In [8]:
person2_scaling = person2_table[person2_table.experiment=='scaling']
display(person2_scaling[['dataset','query_id','retrieved_context_quality','answer_quality',
                        'irrelevant_chunk_fraction','retrieval_ms','generation_ms','end_to_end_ms']])
display(pd.read_csv(person2_run/'scaling_summary.csv'))
display(pd.read_csv(person2_run/'scaling_pairs.csv'))
# Full retrieved chunks and actual answers for every scaling request.
for row in person2_scaling.to_dict('records'):
    print('\n'+'='*72+'\n'+row['row_id']+'\n'+row['query'])
    for hit in json.loads(row['retrieved_chunks']):
        print(f"\nRank {hit['rank']} | {hit['chunk_id']} | cosine={hit['score']:.4f}")
        print('Paragraphs:',', '.join(hit['paragraph_ids']))
        print(hit['text'])
    print('\nActual answer:',row['answer'])
display(Markdown((root/'a8/person2/FINDINGS.md').read_text()))

,dataset,query_id,retrieved_context_quality,answer_quality,irrelevant_chunk_fraction,retrieval_ms,generation_ms,end_to_end_ms
15,small,Q1,100.0,0.0,0.333333,35.797030,21575.341641,21611.266888
16,small,Q2,100.0,0.0,0.333333,40.145317,12551.118736,12591.351337
17,small,Q3,100.0,0.0,0.666667,39.254688,35927.502920,35966.847196
18,small,Q4,100.0,75.0,0.333333,32.831625,42192.313389,42225.305105
19,small,Q5,100.0,100.0,0.333333,49.610605,42861.613197,42911.330076
20,full,Q1,100.0,0.0,0.333333,32.583845,21924.268021,21956.973924
21,full,Q2,100.0,0.0,0.333333,38.908397,13272.250341,13311.250360
22,full,Q3,100.0,25.0,0.666667,38.288567,31778.610197,31816.993325
23,full,Q4,100.0,75.0,0.333333,31.130162,41625.889791,41657.134826
24,full,Q5,100.0,100.0,0.333333,35.846120,48646.321702,48682.277438


,dataset,queries,mean_context_quality,mean_answer_quality,mean_irrelevant_chunk_fraction,median_retrieval_ms,median_generation_ms,median_end_to_end_ms
0,small,5,100.0,35.0,0.4,39.254688,35927.502920,35966.847196
1,full,5,100.0,40.0,0.4,35.846120,31778.610197,31816.993325


,query_id,small_answer_quality,full_answer_quality,answer_quality_delta,small_noise_fraction,full_noise_fraction,small_end_to_end_ms,full_end_to_end_ms
0,Q1,0.0,0.0,0.0,0.333333,0.333333,21611.266888,21956.973924
1,Q2,0.0,0.0,0.0,0.333333,0.333333,12591.351337,13311.250360
2,Q3,0.0,25.0,25.0,0.666667,0.666667,35966.847196,31816.993325
3,Q4,75.0,75.0,0.0,0.333333,0.333333,42225.305105,41657.134826
4,Q5,100.0,100.0,0.0,0.333333,0.333333,42911.330076,48682.277438



small/small/hybrid/Q1
I finished closing the kitchen, but the dishwasher is still running. What should I mark on the board and what should I do with the dishwasher?

Rank 1 | hybrid-chores-01 | cosine=0.6674
Paragraphs: chores-01, chores-02
The kitchen closer takes the evening handoff after the last meal. The schedule assigns one closer for each evening and lists a backup for weeks when a resident travels. Before starting, the closer checks the task board for notes about items soaking or food waiting to cool. A note should identify the item and the resident who will finish it. The closer can complete the rest of the routine while that resident handles the exception. Clean cookware belongs in the drying rack until it is dry enough to put away. Keep the main counter available for breakfast preparation.

When the evening kitchen close is finished, mark the existing Kitchen Close card complete on the household task board. If the dishwasher is running, clip the Running tag to its handle an

# Person 2 — Assignment 8 results

Completed November 9, 2026 using the returned Google Colab CPU run. All 15 small-model matrix requests and 10 scaling requests contain actual answers, full retrieved chunks, exact prompts and measured end-to-end latency. Quality and relevance judgments are AI-assisted; student signoff is not recorded.

## Configuration and evidence

- Encoder: `sentence-transformers/all-MiniLM-L6-v2`, revision `1110a243fdf4706b3f48f1d95db1a4f5529b4d41`, 384 dimensions, normalized float32 cosine search, k=3.
- Generator: `Qwen/Qwen2-0.5B-Instruct`, revision `c540970f9e29518b1d8f06ab8b24cba66ad77b6d`, previously evaluated in A6/A7. The unchanged Person 3 system prompt, cited-context format, greedy decoding, 128-token cap and CPU float32 calculation are used throughout.
- Runtime: hosted Linux x86-64, Python 3.13.18, torch 2.14.1+cpu, NumPy 2.4.6, sentence-transformers 5.1.2, transformers 4.57.6, four PyTorch threads. Matrix/scaling manifests match on controls and environment.
- Full corpus: 30 paragraphs, 2,901 words. Small subset: 15 paragraphs, 1,461 words. The shared source/chunkers are unchanged. Both datasets retain all five reference answers.
- Raw evidence: `../results/person2/completed-2026-11-09/`; every review is SHA-256-bound to its entire raw row, including prompt, answer, hits and timing. All three downloaded ZIP snapshots agree on shared raw files; the executed Colab notebook contains no error outputs.

## Evaluation method

Context quality is the shared rubric’s percentage of reference quotes present in the top three retrieved texts. Source recall and reciprocal rank are separate diagnostics, not semantic precision. Answer quality awards each fully conveyed required fact, subtracts one fact point per distinct materially incorrect or unsupported instruction, floors at zero, then divides by the number of required facts. Compound facts require all parts. Exact fact indices, deductions and rationale are in `reviews.csv`.

Noise is separately reviewed: a chunk is relevant if it answers a requested part or supplies directly applicable conditions, handoff or approval context. A broad household vocabulary match is insufficient. Redundant but relevant passages are not automatically noise; a different source ID is not automatically noise. Noise fraction is `(3 - relevant_hits) / 3`.

## Fifteen-row evaluation table

All rows below use the MiniLM encoder and fixed Qwen generator. Full questions, answers, complete retrieved chunks and longer review notes are in `matrix_evaluation.csv`; raw JSONL also retains exact prompts. Latency is observed warmed request time from query encoding through decoded text, excluding downloads, index construction, warm-up and review.

| Strategy | Query | Context /100 | Answer /100 | Request seconds | Notes |
|---|---|---:|---:|---:|---|
| fixed | Q1 | 100 | 66.67 | 29.95 | Tag attachment omitted |
| fixed | Q2 | 100 | 0.00 | 11.52 | Incorrectly closes unpurchased entry |
| fixed | Q3 | 100 | 0.00 | 36.34 | Missing facts; invented log fields |
| fixed | Q4 | 100 | 0.00 | 40.02 | Conditional notification wrong; capped |
| fixed | Q5 | 100 | 20.00 | 25.93 | Missing placements/lamp/completion rule |
| overlap | Q1 | 100 | 100.00 | 41.11 | All handoff facts |
| overlap | Q2 | 100 | 0.00 | 14.65 | Quantity alone does not answer question |
| overlap | Q3 | 100 | 0.00 | 33.55 | Misuses appliance upkeep as drill context |
| overlap | Q4 | 100 | 75.00 | 37.00 | Omits Plumbing/Maintenance names |
| overlap | Q5 | 100 | 100.00 | 45.55 | All required facts before cap |
| hybrid | Q1 | 100 | 0.00 | 21.84 | Invents status card and service call |
| hybrid | Q2 | 100 | 0.00 | 13.89 | Describes a purchase despite no substitute |
| hybrid | Q3 | 100 | 25.00 | 31.52 | Correct kit bin; missing name/closure |
| hybrid | Q4 | 100 | 75.00 | 40.41 | Omits Plumbing/Maintenance names |
| hybrid | Q5 | 100 | 100.00 | 41.37 | All required facts before cap |

Query mapping: Q1 kitchen close/running dishwasher; Q2 unavailable grocery/no approved replacement; Q3 drill location/checkout/return; Q4 slow-drip reporting; Q5 guest-room readiness. The five verbatim queries and required facts remain in `../data/queries.json`.

| Strategy | Mean context | Mean answer | Median request seconds |
|---|---:|---:|---:|
| fixed | 100.00 | 17.33 | 29.95 |
| overlap | 100.00 | 55.00 | 37.00 |
| hybrid | 100.00 | 40.00 | 31.52 |

All 15 contexts contain every reference quote, yet the mean answer score is only 37.44. Retrieval availability does not guarantee the generator uses the evidence. Overlap performs best on this small-model slice; this is not the team-wide chunking comparison. Three matrix requests reach the output cap; the guest answers contain all required facts before the cut, whereas the fixed tap answer is incomplete. Score the text actually returned, not a presumed continuation.

## Controlled dataset scaling

Only dataset content changes within this experiment. Small model, hybrid chunking, k=3, queries, generator, prompt, decoding, cap, thread setting and software environment stay fixed. Small/full hybrid indexes have 10/20 chunks. Their raw float32 vector payloads are 15,360/30,720 bytes, calculated as chunks × 384 × 4; these exclude metadata, text, compression and model weights.

| Dataset | Evidence quality | Mean answer | Reviewed noise | Median retrieval ms | Median generation s | Median full request s |
|---|---:|---:|---:|---:|---:|---:|
| small | 100 | 35.00 | 40% | 39.25 | 35.93 | 35.97 |
| full | 100 | 40.00 | 40% | 35.85 | 31.78 | 31.82 |

| Query | Small answer /100 | Full answer /100 | Difference | Small/full noise |
|---|---:|---:|---:|---|
| Q1 | 0.00 | 0.00 | +0.00 | 33% / 33% |
| Q2 | 0.00 | 0.00 | +0.00 | 33% / 33% |
| Q3 | 0.00 | 25.00 | +25.00 | 67% / 67% |
| Q4 | 75.00 | 75.00 | +0.00 | 33% / 33% |
| Q5 | 100.00 | 100.00 | +0.00 | 33% / 33% |

**Retrieval quality:** both sizes keep the supporting source at rank one and all reference evidence within the top three. Added paragraphs change lower-ranked hits for Q1, Q2, Q3 and Q5; Q4 keeps the same hits. There is no evidence-coverage loss from this controlled distractor addition.

**Noise:** the reviewed irrelevant fraction stays 40% overall. Extra topics sometimes replace existing irrelevant context rather than adding more irrelevant hits at fixed k. The inventory substitution-approval clause is relevant to Q2 even though it supplies no answer quote; the refund passage is not. Q5’s guest-supply guidance is relevant, whereas arrival/departure coordination does not establish readiness. These judgments are disclosed and may be revised after student review.

**Answer quality:** the five-point mean increase comes entirely from Q3 (0→25): the small-set answer invents a green kitchen bin for the kit; the full-set answer returns the kit to the correct blue hallway bin but still omits the borrower name and entry closure. Both Q1 answers invent repair instructions despite complete handoff evidence. Larger data did not eliminate generation errors.

**Scoring sensitivity:** the small-set Q2 says “checked out” rather than “unchecked.” The base review credits Out of stock, withholds the correct-status fact and deducts one point for the contradictory status instruction, yielding zero. If a reviewer treats that phrase only as an omission and removes the deduction, this row becomes 50 and the small-set mean becomes 45 rather than 35. That reverses the aggregate ordering; the observed five-point advantage is not a robust general scaling benefit.

**Latency:** the full set is not consistently faster per query. These are five warmed requests per dataset, run in dataset order rather than a randomized repeated load test. Variable output length and hosted CPU conditions affect generation; generation dominates this tiny exact-search index. Small changes in median retrieval latency do not establish a scaling speed advantage. No 10K-user/day throughput claim follows.

## Integration and limitations

- These 25 rows complete the Person 2 experiment slice. Person 3’s separate 45-row comparison remains the matched cross-encoder evidence; do not pool Colab and Mac timing medians.
- Person 4 owns the combined chunking comparison, failure/improvement section, cost/design reflection and final team ZIP. The observed unsupported service call, out-of-stock failures and invented kit location are available for that analysis.
- The original 2026-11-08 Mac retrieval-only measurements and indexes remain untouched. See `RETRIEVAL_FINDINGS_2026-11-08.md`; their timings are not relabeled as Colab measurements.
- Exact prompts confirm no answer key entered generation. There is no context truncation. Five Colab requests in total reach the 128-token cap (three matrix, two scaling).
- Quality/relevance reviews are AI-assisted, not independent human adjudication. Student signoff is not recorded.


## Person 4 · Failure analysis, improvement, and system design

Parts **3–4** (three failures, top-k improvement before/after, cost awareness,
RAG vs fine-tuning vs prompting, and the 10K-user design) are written in
[`rag_analysis.md`](rag_analysis.md). Saved top-k evidence:
`results/person4/improvement_topk2_large_hybrid_Q1.json`.
Validate with `python -m a8.person4_improvement`.
